In [ ]:

# ── PEN OIS Fair Value Model ──────────────────────────────────────────────────
# Single notebook: ipywidgets + plotly. ION compounding (TEA), ACT/360, ModFol.

import numpy as np
import pandas as pd
from datetime import date, timedelta
from scipy.optimize import minimize
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import warnings
warnings.filterwarnings("ignore")

# ── Config ────────────────────────────────────────────────────────────────────
DATA_PATH = "ois_history.xlsx"  # Sheet1, cols: dates, 3mo, 6mo, 9mo, 12mo (TEA)
TENOR_LABELS = ["3M", "6M", "9M", "12M"]
TENOR_MONTHS = [3, 6, 9, 12]
DEFAULT_NOTIONAL = 10_000_000  # 10M PEN
MAXENT_TOL = 0.1  # 0.1 bp tolerance for MaxEnt relaxation

# ── PE Holidays (fixed + Easter-dependent) ────────────────────────────────────
def pe_holidays(year):
    "Return set of PE holiday dates for a given year."
    fixed = [
        date(year, 1, 1), date(year, 5, 1), date(year, 6, 29),
        date(year, 7, 28), date(year, 7, 29), date(year, 8, 30),
        date(year, 10, 8), date(year, 11, 1), date(year, 12, 8), date(year, 12, 25),
    ]
    # Easter (Anonymous Gregorian algorithm)
    a = year % 19
    b, c = divmod(year, 100)
    d, e = divmod(b, 4)
    f = (b + 8) // 25
    g = (b - f + 1) // 3
    h = (19 * a + b - d - g + 15) % 30
    i, k = divmod(c, 4)
    l = (32 + 2 * e + 2 * i - h - k) % 7
    m = (a + 11 * h + 22 * l) // 451
    month = (h + l - 7 * m + 114) // 31
    day = ((h + l - 7 * m + 114) % 31) + 1
    easter = date(year, month, day)
    holy_thu = easter - timedelta(days=3)
    good_fri = easter - timedelta(days=2)
    return set(fixed + [holy_thu, good_fri])

# Build holiday set covering 2025-2028
ALL_HOLIDAYS = set()
for y in range(2025, 2029):
    ALL_HOLIDAYS |= pe_holidays(y)

def is_business_day(d):
    return d.weekday() < 5 and d not in ALL_HOLIDAYS

def next_bus_day(d):
    d = d + timedelta(days=1)
    while not is_business_day(d):
        d = d + timedelta(days=1)
    return d

def mod_fol(d):
    "Modified Following: roll forward, but if month changes roll backward."
    orig_month = d.month
    adj = d
    while not is_business_day(adj):
        adj = adj + timedelta(days=1)
    if adj.month != orig_month:
        adj = d
        while not is_business_day(adj):
            adj = adj - timedelta(days=1)
    return adj

# ── BCRP Meeting Dates ────────────────────────────────────────────────────────
# 2026 confirmed. Rate change effective M+1 (next calendar day after meeting).
MEETINGS_2026 = [
    date(2026, 1, 8), date(2026, 2, 12), date(2026, 3, 12), date(2026, 4, 9),
    date(2026, 5, 14), date(2026, 6, 11), date(2026, 7, 9), date(2026, 8, 13),
    date(2026, 9, 10), date(2026, 10, 7), date(2026, 11, 12), date(2026, 12, 10),
]

def second_thursday(year, month):
    "2nd Thursday of a month."
    d = date(year, month, 1)
    # find first thursday
    d = d + timedelta(days=(3 - d.weekday()) % 7)
    return d + timedelta(days=7)

def build_meeting_schedule(val_date, n=14):
    "Return list of next n meeting dates from val_date onward."
    # Start with 2026 confirmed meetings
    all_meetings = sorted(MEETINGS_2026)
    # Generate 2027+ as 2nd Thursday
    y = 2027
    while len(all_meetings) < 50:
        for m in range(1, 13):
            all_meetings.append(second_thursday(y, m))
        y += 1
    all_meetings = sorted(all_meetings)
    future = [m for m in all_meetings if m > val_date]
    return future[:n]

# ── Default Inputs ────────────────────────────────────────────────────────────
DEFAULT_TIBO_TEA = 4.25  # %
DEFAULT_OIS_TNA = {"3M": 4.18, "6M": 4.1965, "9M": 4.1950, "12M": 4.095}

# ── Default Paths (bp change per meeting) ─────────────────────────────────────
# Keys = path name, values = list of bp changes at each of the next 14 meetings
def make_path(changes_dict):
    "changes_dict: {meeting_index (0-based): bp_change}. Returns list of 14 zeros with overrides."
    p = [0] * 14
    for k, v in changes_dict.items():
        if k < 14:
            p[k] = v
    return p

# Meeting indices (0-based from next meeting after today):
# For Mar 19, 2026: next meetings are Apr9=0, May14=1, Jun11=2, Jul9=3, Aug13=4,
# Sep10=5, Oct7=6, Nov12=7, Dec10=8, Jan27=9, Feb27=10, Mar27=11, ...
DEFAULT_PATHS = {
    "Hold":           make_path({}),
    "Apr-25":         make_path({0: -25}),
    "May-25":         make_path({1: -25}),
    "Jun-25":         make_path({2: -25}),
    "Jul-25":         make_path({3: -25}),
    "Apr/Jul-25":     make_path({0: -25, 3: -25}),
    "Apr/Jun-25":     make_path({0: -25, 2: -25}),
    "Jun/Sep-25":     make_path({2: -25, 5: -25}),
    "Apr/Jul/Oct-25": make_path({0: -25, 3: -25, 6: -25}),
    "Apr-50":         make_path({0: -50}),
    "Apr/May/Jun-25": make_path({0: -25, 1: -25, 2: -25}),
    "Apr+25":         make_path({0: 25}),
}

DEFAULT_WEIGHTS = {name: 100.0 / len(DEFAULT_PATHS) for name in DEFAULT_PATHS}

print("Cell 1 loaded: config, holidays, meetings, defaults.")


In [ ]:

# ── Pure Functions ────────────────────────────────────────────────────────────

# ── Settlement & maturity dates ───────────────────────────────────────────────
def settle_date(val_date):
    "T+2 settlement: advance 2 business days from val_date."
    d = val_date
    for _ in range(2):
        d = next_bus_day(d)
    return d

def maturity_date(settle, months):
    "Maturity = settle + N months, adjusted ModFol."
    y = settle.year + (settle.month - 1 + months) // 12
    m = (settle.month - 1 + months) % 12 + 1
    d = min(settle.day, [31,29 if y%4==0 and (y%100!=0 or y%400==0) else 28,31,30,31,30,31,31,30,31,30,31][m-1])
    raw = date(y, m, d)
    return mod_fol(raw)

# ── Rate path builder ─────────────────────────────────────────────────────────
def build_rate_path(tibo_tea, bp_changes, meetings, start, end):
    """Build daily TIBO (TEA %) series from start to end (exclusive).
    bp_changes: list of bp changes at each meeting. Rate changes effective M+1.
    Returns list of (date, tibo_tea) for each calendar day in [start, end)."""
    # Build effective date -> rate map
    # Meetings are post-close, so rate changes effective day after meeting
    rate = tibo_tea
    change_dates = {}
    for i, mtg in enumerate(meetings):
        if i < len(bp_changes) and bp_changes[i] != 0:
            eff = mtg + timedelta(days=1)
            change_dates[eff] = bp_changes[i]

    path = []
    current_rate = tibo_tea
    d = start
    while d < end:
        if d in change_dates:
            current_rate += change_dates[d] / 100  # bp to %
        path.append((d, current_rate))
        d += timedelta(days=1)
    return path

# ── ION pricing (TEA compounding) ────────────────────────────────────────────
def price_ois_float(tibo_tea, bp_changes, meetings, start, end):
    """Compute OIS float leg value using ION compounding.
    Block factor for rate r_j (TEA%) over m_j calendar days: (1 + r_j/100)^(m_j/360)
    Float = product of all block factors."""
    rate_path = build_rate_path(tibo_tea, bp_changes, meetings, start, end)

    # Group consecutive days with same rate into blocks
    blocks = []
    if not rate_path:
        return 1.0
    cur_rate = rate_path[0][1]
    cur_days = 0
    for _, r in rate_path:
        if abs(r - cur_rate) < 1e-12:
            cur_days += 1
        else:
            blocks.append((cur_rate, cur_days))
            cur_rate = r
            cur_days = 1
    blocks.append((cur_rate, cur_days))

    # Float = product of block factors
    float_val = 1.0
    for r_tea_pct, m_days in blocks:
        float_val *= (1 + r_tea_pct / 100) ** (m_days / 360)
    return float_val

def ois_fv_tna(tibo_tea, bp_changes, meetings, val_date, months):
    "Compute OIS fair value rate (TNA%) for a given tenor."
    sett = settle_date(val_date)
    mat = maturity_date(sett, months)
    D = (mat - sett).days
    float_val = price_ois_float(tibo_tea, bp_changes, meetings, sett, mat)
    return (float_val - 1) * 360 / D * 100  # TNA %

def compute_all_fvs(tibo_tea, paths_dict, meetings, val_date):
    "Compute FV (TNA%) for all paths and all tenors. Returns dict {path_name: {tenor: fv}}."
    result = {}
    for name, bp_chg in paths_dict.items():
        result[name] = {}
        for label, months in zip(TENOR_LABELS, TENOR_MONTHS):
            result[name][label] = ois_fv_tna(tibo_tea, bp_chg, meetings, val_date, months)
    return result

# ── DV01 ──────────────────────────────────────────────────────────────────────
def dv01(notional, rate_tna_pct, days):
    "DV01 = N * 0.0001 * (D/360) / (1 + K*D/360). K = rate in decimal."
    K = rate_tna_pct / 100
    return notional * 0.0001 * (days / 360) / (1 + K * days / 360)

def compute_dv01s(mkt_rates, val_date, notional=DEFAULT_NOTIONAL):
    "Compute DV01 for each tenor at market rates."
    sett = settle_date(val_date)
    result = {}
    for label, months in zip(TENOR_LABELS, TENOR_MONTHS):
        mat = maturity_date(sett, months)
        D = (mat - sett).days
        result[label] = dv01(notional, mkt_rates[label], D)
    return result

def tenor_days(val_date):
    "Return dict {tenor: calendar days} for each tenor."
    sett = settle_date(val_date)
    return {label: (maturity_date(sett, m) - sett).days for label, m in zip(TENOR_LABELS, TENOR_MONTHS)}

# ── Spread & Butterfly calcs ─────────────────────────────────────────────────
def calc_spreads(rates):
    "Compute upper-triangle spread matrix (bps). rates = dict {tenor: rate_tna%}."
    n = len(TENOR_LABELS)
    spreads = {}
    for i in range(n):
        for j in range(i + 1, n):
            key = f"{TENOR_LABELS[i]}/{TENOR_LABELS[j]}"
            spreads[key] = (rates[TENOR_LABELS[j]] - rates[TENOR_LABELS[i]]) * 100  # bps
    return spreads

BUTTERFLY_DEFS = {
    "Front(3M,6M,9M)":  ("3M", "6M", "9M"),
    "Back(6M,9M,12M)":  ("6M", "9M", "12M"),
    "Wide6M(3M,6M,12M)": ("3M", "6M", "12M"),
    "Wide9M(3M,9M,12M)": ("3M", "9M", "12M"),
}

def calc_butterflies(rates):
    "Compute butterfly values (bps). Fly = 2*Body - Wing1 - Wing2."
    result = {}
    for name, (w1, body, w2) in BUTTERFLY_DEFS.items():
        fly = (2 * rates[body] - rates[w1] - rates[w2]) * 100  # bps
        result[name] = fly
    return result

# ── Hedge solver ──────────────────────────────────────────────────────────────
def solve_hedge_spread(dv01s, leg1, leg2, dir1, dir2, anchor_leg, anchor_notional, target_dv01=0):
    """Solve for hedge leg notional in a spread trade.
    dir1, dir2: +1 for receiver, -1 for payer.
    Returns dict with notionals and net DV01."""
    if anchor_leg == leg1:
        # solve for leg2 notional
        # dir1*dv01_1(N1) + dir2*dv01_2(N2) = target_dv01
        # dv01 scales linearly with notional
        dv01_1 = dv01s[leg1] * dir1
        dv01_per_unit_2 = dv01s[leg2] / DEFAULT_NOTIONAL * dir2
        needed = target_dv01 - dv01_1
        n2 = needed / dv01_per_unit_2 if abs(dv01_per_unit_2) > 1e-12 else 0
        return {leg1: anchor_notional, leg2: abs(n2), "net_dv01": dv01_1 + dv01_per_unit_2 * n2}
    else:
        dv01_2 = dv01s[leg2] * dir2
        dv01_per_unit_1 = dv01s[leg1] / DEFAULT_NOTIONAL * dir1
        needed = target_dv01 - dv01_2
        n1 = needed / dv01_per_unit_1 if abs(dv01_per_unit_1) > 1e-12 else 0
        return {leg1: abs(n1), leg2: anchor_notional, "net_dv01": dv01_per_unit_1 * n1 + dv01_2}

def solve_hedge_butterfly(dv01s, w1, body, w2, body_notional, target_dv01=0):
    """Solve butterfly hedge. Body notional given, wing notionals split proportional to wing DV01.
    Convention: receive body, pay wings (or vice versa)."""
    dv01_body = dv01s[body]
    dv01_w1 = dv01s[w1]
    dv01_w2 = dv01s[w2]
    # total wing dv01 per unit notional
    total_body_dv01 = dv01_body * body_notional / DEFAULT_NOTIONAL
    # split wings proportional to their dv01
    total_wing_dv01_per_unit = dv01_w1 + dv01_w2
    if abs(total_wing_dv01_per_unit) < 1e-12:
        return {w1: 0, body: body_notional, w2: 0}
    # each wing gets share of body dv01
    wing_total_notional = abs(total_body_dv01 - target_dv01) / (total_wing_dv01_per_unit / DEFAULT_NOTIONAL)
    n_w1 = wing_total_notional * (dv01_w1 / total_wing_dv01_per_unit)
    n_w2 = wing_total_notional * (dv01_w2 / total_wing_dv01_per_unit)
    return {w1: abs(n_w1), body: body_notional, w2: abs(n_w2)}

# ── MaxEnt ────────────────────────────────────────────────────────────────────
def maxent_weights(fv_matrix, mkt_rates, tol_bp=MAXENT_TOL):
    """Maximum entropy optimization over user-defined paths.
    fv_matrix: array (n_paths, n_tenors) of FV rates (TNA%).
    mkt_rates: array (n_tenors,) of market rates (TNA%).
    Returns: weight array (n_paths,)."""
    n = fv_matrix.shape[0]
    if n == 0:
        return np.array([])

    mkt = np.array(mkt_rates)
    tol = tol_bp / 100  # convert bp to % tolerance

    def neg_entropy(w):
        # avoid log(0)
        w_safe = np.clip(w, 1e-15, None)
        return np.sum(w_safe * np.log(w_safe))

    def neg_entropy_jac(w):
        w_safe = np.clip(w, 1e-15, None)
        return np.log(w_safe) + 1

    constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1}]

    # Try equality constraints first, then relax
    for current_tol in [0, tol]:
        cons = list(constraints)
        for j in range(fv_matrix.shape[1]):
            weighted_rate_j = j  # capture
            if current_tol == 0:
                cons.append({"type": "eq", "fun": lambda w, jj=j: w @ fv_matrix[:, jj] - mkt[jj]})
            else:
                cons.append({"type": "ineq", "fun": lambda w, jj=j: current_tol - abs(w @ fv_matrix[:, jj] - mkt[jj])})

        w0 = np.ones(n) / n
        bounds = [(1e-10, 1)] * n
        res = minimize(neg_entropy, w0, jac=neg_entropy_jac, method="SLSQP",
                       bounds=bounds, constraints=cons, options={"maxiter": 1000, "ftol": 1e-12})
        if res.success:
            w = res.x / res.x.sum()
            return w

    # Fallback: return uniform
    return np.ones(n) / n

def maxent_meeting_probs(tibo_tea, weights, paths_dict, meetings, val_date):
    """Compute per-meeting P(Cut)/P(Hold)/P(Hike) and expected/cumulative move.
    Returns DataFrame."""
    path_names = list(paths_dict.keys())
    bp_changes = np.array([paths_dict[n] for n in path_names])  # (n_paths, n_meetings)
    w = np.array([weights.get(n, 0) for n in path_names])
    w = w / w.sum() if w.sum() > 0 else w

    n_mtg = min(bp_changes.shape[1], len(meetings))
    rows = []
    cum_move = 0
    for i in range(n_mtg):
        changes_i = bp_changes[:, i]
        p_cut = w[changes_i < 0].sum()
        p_hold = w[changes_i == 0].sum()
        p_hike = w[changes_i > 0].sum()
        exp_move = (w * changes_i).sum()
        cum_move += exp_move
        rows.append({
            "Meeting": meetings[i].strftime("%d-%b-%y"),
            "P(Cut)%": round(p_cut * 100, 1),
            "P(Hold)%": round(p_hold * 100, 1),
            "P(Hike)%": round(p_hike * 100, 1),
            "ExpMove(bp)": round(exp_move, 2),
            "CumMove(bp)": round(cum_move, 2),
        })
    return pd.DataFrame(rows)

# ── TEA ↔ TNA conversion ─────────────────────────────────────────────────────
def tea_to_tna(r_tea_pct):
    "Convert TEA% to TNA%: r_tna = ((1 + r_tea/100)^(1/360) - 1) * 360 * 100"
    return ((1 + r_tea_pct / 100) ** (1 / 360) - 1) * 360 * 100

def tna_to_tea(r_tna_pct):
    "Convert TNA% to TEA%: r_tea = ((1 + r_tna/(360*100))^360 - 1) * 100"
    return ((1 + r_tna_pct / (360 * 100)) ** 360 - 1) * 100

# ── Synthetic history for dev ─────────────────────────────────────────────────
def generate_synthetic_history(val_date, lookback_days=365):
    "Generate 1Y synthetic OIS history (TEA) with ~2-3bp daily vol, trending down ~50bp."
    np.random.seed(42)
    dates = pd.date_range(end=val_date, periods=lookback_days, freq="B")
    start_rates = {"3mo": 4.75, "6mo": 4.70, "9mo": 4.65, "12mo": 4.60}
    drift = -50 / lookback_days / 100  # -50bp over 1Y, in pct per day
    data = {"dates": dates}
    for col, start in start_rates.items():
        noise = np.random.normal(0, 2.5 / 100, lookback_days).cumsum()  # ~2.5bp daily vol
        trend = np.arange(lookback_days) * drift
        data[col] = start + trend + noise
    return pd.DataFrame(data)

def load_or_generate_history(val_date):
    "Try loading xlsx, fall back to synthetic."
    try:
        df = pd.read_excel(DATA_PATH, sheet_name="Sheet1")
        df.columns = [c.strip().lower() for c in df.columns]
        return df
    except Exception:
        return generate_synthetic_history(val_date)

print("Cell 2 loaded: pricing, DV01, spreads, butterflies, MaxEnt, history.")


In [ ]:

# ── Widgets + Display ─────────────────────────────────────────────────────────

# ── State container ───────────────────────────────────────────────────────────
state = {
    "val_date": date.today(),
    "tibo_tea": DEFAULT_TIBO_TEA,
    "mkt_rates": dict(DEFAULT_OIS_TNA),
    "paths": {k: list(v) for k, v in DEFAULT_PATHS.items()},
    "weights": dict(DEFAULT_WEIGHTS),
    "notional": DEFAULT_NOTIONAL,
}

# ── CSS ───────────────────────────────────────────────────────────────────────
display(HTML("""<style>
.fv-table { border-collapse:collapse; font-size:12px; font-family:monospace; }
.fv-table td, .fv-table th { padding:3px 8px; border:1px solid #ccc; text-align:right; }
.fv-table th { background:#f0f0f0; text-align:center; }
.cell-green { background:#c6efce; color:#006100; }
.cell-red { background:#ffc7ce; color:#9c0006; }
.row-mkt { background:#cce5ff; }
.row-fv { background:#fff9c4; }
.cell-bold-gray { font-weight:bold; color:#555; }
</style>"""))

# ── Input widgets ─────────────────────────────────────────────────────────────
w_valdate = widgets.DatePicker(description="ValDate:", value=date.today(),
                               style={"description_width": "60px"})
w_tibo = widgets.FloatText(description="TIBO(TEA%):", value=DEFAULT_TIBO_TEA, step=0.01,
                           style={"description_width": "80px"}, layout=widgets.Layout(width="180px"))

w_ois = {}
for t in TENOR_LABELS:
    w_ois[t] = widgets.FloatText(description=f"OIS {t}:", value=DEFAULT_OIS_TNA[t], step=0.0005,
                                  style={"description_width": "60px"}, layout=widgets.Layout(width="180px"))

w_notional = widgets.FloatText(description="Notional:", value=DEFAULT_NOTIONAL, step=1e6,
                                style={"description_width": "70px"}, layout=widgets.Layout(width="200px"))

# ── Path editor widgets ──────────────────────────────────────────────────────
path_out = widgets.Output()
MAX_MTG = 14

def build_path_grid():
    "Build path editor: name + bp changes per meeting + weight."
    rows = []
    for name, bp_list in state["paths"].items():
        name_w = widgets.Text(value=name, layout=widgets.Layout(width="120px"))
        bp_ws = [widgets.IntText(value=bp_list[i] if i < len(bp_list) else 0,
                                  layout=widgets.Layout(width="45px"))
                 for i in range(MAX_MTG)]
        wt_w = widgets.FloatText(value=state["weights"].get(name, 0),
                                  layout=widgets.Layout(width="60px"), step=1)
        rows.append({"name": name_w, "bps": bp_ws, "wt": wt_w, "orig_name": name})
    return rows

path_rows = build_path_grid()

def add_path_clicked(b):
    new_name = f"Path{len(state['paths']) + 1}"
    state["paths"][new_name] = [0] * MAX_MTG
    state["weights"][new_name] = 0
    refresh_path_editor()

def remove_path_clicked(b):
    if len(state["paths"]) > 1:
        last = list(state["paths"].keys())[-1]
        del state["paths"][last]
        if last in state["weights"]:
            del state["weights"][last]
        refresh_path_editor()

def read_paths_from_widgets():
    "Sync path_rows widgets back into state."
    new_paths = {}
    new_weights = {}
    for row in path_rows:
        name = row["name"].value.strip() or row["orig_name"]
        bps = [w.value for w in row["bps"]]
        new_paths[name] = bps
        new_weights[name] = row["wt"].value
    state["paths"] = new_paths
    state["weights"] = new_weights

def refresh_path_editor():
    global path_rows
    path_rows = build_path_grid()
    with path_out:
        clear_output(wait=True)
        render_path_editor()

def render_path_editor():
    meetings = build_meeting_schedule(state["val_date"], MAX_MTG)
    mtg_labels = [m.strftime("%b%d") for m in meetings]
    header = widgets.HBox([widgets.Label("Name", layout=widgets.Layout(width="120px"))] +
                          [widgets.Label(ml, layout=widgets.Layout(width="45px")) for ml in mtg_labels] +
                          [widgets.Label("Wt%", layout=widgets.Layout(width="60px"))])
    row_boxes = []
    for row in path_rows:
        row_boxes.append(widgets.HBox([row["name"]] + row["bps"] + [row["wt"]]))

    btn_add = widgets.Button(description="+Path", button_style="success",
                              layout=widgets.Layout(width="80px"))
    btn_rem = widgets.Button(description="-Path", button_style="danger",
                              layout=widgets.Layout(width="80px"))
    btn_add.on_click(add_path_clicked)
    btn_rem.on_click(remove_path_clicked)

    display(widgets.VBox([header] + row_boxes + [widgets.HBox([btn_add, btn_rem])]))

# ── Output areas ──────────────────────────────────────────────────────────────
out_fv = widgets.Output()
out_spread = widgets.Output()
out_fly = widgets.Output()
out_dv01 = widgets.Output()
out_maxent = widgets.Output()
out_chart = widgets.Output()

# ── RV dropdown (shared by spread + butterfly) ────────────────────────────────
def rv_options():
    return list(state["paths"].keys()) + ["FV"]

w_rv_select = widgets.Dropdown(options=rv_options(), value="FV", description="RV Path:")

# ── Hedge widgets ─────────────────────────────────────────────────────────────
w_hedge_type = widgets.Dropdown(options=["Spread", "Butterfly"], value="Spread", description="Type:")
w_hedge_leg1 = widgets.Dropdown(options=TENOR_LABELS, value="3M", description="Leg1:")
w_hedge_leg2 = widgets.Dropdown(options=TENOR_LABELS, value="6M", description="Leg2:")
w_hedge_leg3 = widgets.Dropdown(options=TENOR_LABELS, value="12M", description="Wing2:")
w_hedge_dir1 = widgets.Dropdown(options=["Recv(+1)", "Pay(-1)"], value="Recv(+1)", description="Dir1:")
w_hedge_dir2 = widgets.Dropdown(options=["Recv(+1)", "Pay(-1)"], value="Pay(-1)", description="Dir2:")
w_hedge_anchor = widgets.Dropdown(options=TENOR_LABELS, value="3M", description="Anchor:")
w_hedge_anchor_n = widgets.FloatText(value=DEFAULT_NOTIONAL, description="AnchorN:", step=1e6,
                                      layout=widgets.Layout(width="200px"))
w_hedge_target = widgets.FloatText(value=0, description="TargetDV01:", step=100,
                                    layout=widgets.Layout(width="200px"))
out_hedge = widgets.Output()

# ── Chart dropdown ────────────────────────────────────────────────────────────
w_chart_tenor = widgets.Dropdown(options=TENOR_LABELS, value="3M", description="Tenor:")

# ── Master compute ────────────────────────────────────────────────────────────
def compute_and_display(change=None):
    # Read current widget values
    state["val_date"] = w_valdate.value if isinstance(w_valdate.value, date) else date.today()
    state["tibo_tea"] = w_tibo.value
    for t in TENOR_LABELS:
        state["mkt_rates"][t] = w_ois[t].value
    state["notional"] = w_notional.value
    read_paths_from_widgets()

    meetings = build_meeting_schedule(state["val_date"], MAX_MTG)
    mkt = state["mkt_rates"]

    # ── Compute FVs ───────────────────────────────────────────────────────
    all_fvs = compute_all_fvs(state["tibo_tea"], state["paths"], meetings, state["val_date"])

    # Weighted FV
    wt_total = sum(state["weights"].values())
    fv_weighted = {}
    for t in TENOR_LABELS:
        if wt_total > 0:
            fv_weighted[t] = sum(state["weights"].get(n, 0) * all_fvs[n][t]
                                 for n in all_fvs) / wt_total
        else:
            fv_weighted[t] = 0

    # ── 6a. FV Table + Deltas ─────────────────────────────────────────────
    with out_fv:
        clear_output(wait=True)
        html = "<h4>FV Table & Deltas</h4>"
        html += "<table class='fv-table'><tr><th>Path</th><th>Wt%</th>"
        for t in TENOR_LABELS:
            html += f"<th>FV_{t}</th>"
        for t in TENOR_LABELS:
            html += f"<th>Δ_{t}(bp)</th>"
        html += "</tr>"

        # Path rows
        all_fv_values = {t: [] for t in TENOR_LABELS}
        for name in state["paths"]:
            wt = state["weights"].get(name, 0)
            html += f"<tr><td style='text-align:left'>{name}</td><td>{wt:.1f}</td>"
            for t in TENOR_LABELS:
                fv = all_fvs[name][t]
                all_fv_values[t].append(fv)
                html += f"<td>{fv:.4f}</td>"
            for t in TENOR_LABELS:
                delta = (mkt[t] - all_fvs[name][t]) * 100
                css = ""
                if delta > 4:
                    css = " class='cell-green'"
                elif delta < -4:
                    css = " class='cell-red'"
                html += f"<td{css}>{delta:+.2f}</td>"
            html += "</tr>"

        # Mkt row
        html += "<tr class='row-mkt'><td style='text-align:left'><b>Mkt</b></td><td></td>"
        for t in TENOR_LABELS:
            html += f"<td><b>{mkt[t]:.4f}</b></td>"
        for t in TENOR_LABELS:
            html += "<td></td>"
        html += "</tr>"

        # FV row
        html += "<tr class='row-fv'><td style='text-align:left'><b>FV</b></td><td></td>"
        for t in TENOR_LABELS:
            html += f"<td><b>{fv_weighted[t]:.4f}</b></td>"
        for t in TENOR_LABELS:
            d = (mkt[t] - fv_weighted[t]) * 100
            html += f"<td><b>{d:+.2f}</b></td>"
        html += "</tr>"

        # Mkt-FV row
        html += "<tr><td style='text-align:left'>Mkt-FV(bp)</td><td></td>"
        for t in TENOR_LABELS:
            d = (mkt[t] - fv_weighted[t]) * 100
            html += f"<td>{d:+.2f}</td>"
        for t in TENOR_LABELS:
            html += "<td></td>"
        html += "</tr>"

        # HighDelta = Mkt - min(FV), LowDelta = Mkt - max(FV)
        html += "<tr><td style='text-align:left'>HighΔ(Mkt-minFV)</td><td></td>"
        for t in TENOR_LABELS:
            hd = (mkt[t] - min(all_fv_values[t])) * 100
            html += f"<td>{hd:+.2f}</td>"
        for t in TENOR_LABELS:
            html += "<td></td>"
        html += "</tr>"
        html += "<tr><td style='text-align:left'>LowΔ(Mkt-maxFV)</td><td></td>"
        for t in TENOR_LABELS:
            ld = (mkt[t] - max(all_fv_values[t])) * 100
            html += f"<td>{ld:+.2f}</td>"
        for t in TENOR_LABELS:
            html += "<td></td>"
        html += "</tr>"

        html += "</table>"
        display(HTML(html))

    # ── RV calcs (shared) ─────────────────────────────────────────────────
    rv_sel = w_rv_select.value
    if rv_sel == "FV":
        rv_rates = fv_weighted
    elif rv_sel in all_fvs:
        rv_rates = all_fvs[rv_sel]
    else:
        rv_rates = fv_weighted

    mkt_spreads = calc_spreads(mkt)
    rv_spreads = calc_spreads(rv_rates)
    mkt_flies = calc_butterflies(mkt)
    rv_flies = calc_butterflies(rv_rates)

    # ── 6b. Spread RV Matrix ─────────────────────────────────────────────
    with out_spread:
        clear_output(wait=True)
        html = f"<h4>Spread RV Matrix (path: {rv_sel})</h4>"
        html += "<table class='fv-table'><tr><th></th>"
        for t in TENOR_LABELS[1:]:
            html += f"<th>{t}</th>"
        html += "</tr>"
        for i, t1 in enumerate(TENOR_LABELS[:-1]):
            html += f"<tr><th>{t1}</th>"
            for j, t2 in enumerate(TENOR_LABELS[1:]):
                if j < i:
                    html += "<td></td>"
                else:
                    key = f"{t1}/{t2}"
                    ps = rv_spreads.get(key, 0)
                    ms = mkt_spreads.get(key, 0)
                    pickup = ms - ps
                    css = " class='cell-bold-gray'" if abs(pickup) > 4 else ""
                    html += f"<td{css}>{ps:.1f} ({pickup:+.1f}) / {ms:.1f}</td>"
            html += "</tr>"
        html += "</table>"
        html += "<p style='font-size:11px'>+pickup=Recv long/Pay short | -pickup=Pay long/Recv short</p>"
        display(HTML(html))

    # ── 6c. Butterfly RV Table ────────────────────────────────────────────
    with out_fly:
        clear_output(wait=True)
        html = f"<h4>Butterfly RV (path: {rv_sel})</h4>"
        html += "<table class='fv-table'><tr><th>Fly</th><th>Path (pickup) / Mkt</th></tr>"
        for fname in BUTTERFLY_DEFS:
            pf = rv_flies.get(fname, 0)
            mf = mkt_flies.get(fname, 0)
            pickup = mf - pf
            css = " class='cell-bold-gray'" if abs(pickup) > 4 else ""
            html += f"<tr><td style='text-align:left'>{fname}</td>"
            html += f"<td{css}>{pf:.1f} ({pickup:+.1f}) / {mf:.1f}</td></tr>"
        html += "</table>"
        display(HTML(html))

    # ── 6d. DV01 + Hedge ─────────────────────────────────────────────────
    with out_dv01:
        clear_output(wait=True)
        td = tenor_days(state["val_date"])
        dv01s = compute_dv01s(mkt, state["val_date"], state["notional"])
        html = "<h4>DV01 Table</h4><table class='fv-table'><tr><th>Tenor</th><th>Days</th>"
        html += "<th>Rate(TNA%)</th><th>DV01(PEN)</th></tr>"
        for t in TENOR_LABELS:
            html += f"<tr><td>{t}</td><td>{td[t]}</td><td>{mkt[t]:.4f}</td>"
            html += f"<td>{dv01s[t]:,.2f}</td></tr>"
        html += "</table>"
        display(HTML(html))

    with out_hedge:
        clear_output(wait=True)
        dv01s_unit = compute_dv01s(mkt, state["val_date"], DEFAULT_NOTIONAL)
        d1 = 1 if "Recv" in w_hedge_dir1.value else -1
        d2 = 1 if "Recv" in w_hedge_dir2.value else -1
        if w_hedge_type.value == "Spread":
            res = solve_hedge_spread(dv01s_unit, w_hedge_leg1.value, w_hedge_leg2.value,
                                     d1, d2, w_hedge_anchor.value, w_hedge_anchor_n.value,
                                     w_hedge_target.value)
            html = "<h4>Hedge Result (Spread)</h4><table class='fv-table'>"
            html += f"<tr><td>{w_hedge_leg1.value}</td><td>{res[w_hedge_leg1.value]:,.0f}</td></tr>"
            html += f"<tr><td>{w_hedge_leg2.value}</td><td>{res[w_hedge_leg2.value]:,.0f}</td></tr>"
            html += f"<tr><td>Net DV01</td><td>{res['net_dv01']:,.2f}</td></tr></table>"
        else:
            w1, body, w2 = w_hedge_leg1.value, w_hedge_leg2.value, w_hedge_leg3.value
            res = solve_hedge_butterfly(dv01s_unit, w1, body, w2,
                                        w_hedge_anchor_n.value, w_hedge_target.value)
            html = "<h4>Hedge Result (Butterfly)</h4><table class='fv-table'>"
            html += f"<tr><td>{w1}(wing)</td><td>{res[w1]:,.0f}</td></tr>"
            html += f"<tr><td>{body}(body)</td><td>{res[body]:,.0f}</td></tr>"
            html += f"<tr><td>{w2}(wing)</td><td>{res[w2]:,.0f}</td></tr></table>"
        display(HTML(html))

    # ── 6e. MaxEnt ────────────────────────────────────────────────────────
    with out_maxent:
        clear_output(wait=True)
        path_names = list(state["paths"].keys())
        fv_mat = np.array([[all_fvs[n][t] for t in TENOR_LABELS] for n in path_names])
        mkt_arr = np.array([mkt[t] for t in TENOR_LABELS])
        me_w = maxent_weights(fv_mat, mkt_arr)

        # Table 1: paths ranked by MaxEnt weight
        html = "<h4>MaxEnt Probabilities</h4>"
        ranked = sorted(zip(path_names, me_w), key=lambda x: -x[1])
        html += "<table class='fv-table'><tr><th>Path</th><th>UserWt%</th><th>MaxEntWt%</th>"
        for t in TENOR_LABELS:
            html += f"<th>FV_{t}</th>"
        html += "</tr>"
        for name, mew in ranked:
            uw = state["weights"].get(name, 0)
            html += f"<tr><td style='text-align:left'>{name}</td>"
            html += f"<td>{uw:.1f}</td><td>{mew*100:.1f}</td>"
            for t in TENOR_LABELS:
                html += f"<td>{all_fvs[name][t]:.4f}</td>"
            html += "</tr>"
        html += "</table><br>"

        # Table 2: meeting probabilities using MaxEnt weights
        me_wts = {n: w for n, w in zip(path_names, me_w)}
        df_probs = maxent_meeting_probs(state["tibo_tea"], me_wts, state["paths"],
                                        meetings, state["val_date"])
        html += "<table class='fv-table'><tr>"
        for col in df_probs.columns:
            html += f"<th>{col}</th>"
        html += "</tr>"
        for _, row in df_probs.iterrows():
            html += "<tr>"
            for col in df_probs.columns:
                val = row[col]
                html += f"<td>{val}</td>"
            html += "</tr>"
        html += "</table>"
        display(HTML(html))

    # ── 7. Plotly Chart ───────────────────────────────────────────────────
    with out_chart:
        clear_output(wait=True)
        tenor_sel = w_chart_tenor.value
        tenor_col_map = {"3M": "3mo", "6M": "6mo", "9M": "9mo", "12M": "12mo"}
        col = tenor_col_map[tenor_sel]

        df_hist = load_or_generate_history(state["val_date"])

        fig = go.Figure()

        # Historical market line (TEA → TNA)
        if col in df_hist.columns:
            tna_hist = df_hist[col].apply(tea_to_tna)
            fig.add_trace(go.Scatter(x=df_hist["dates"], y=tna_hist,
                                     mode="lines", name="Mkt",
                                     line=dict(color="black", dash="dash", width=1.5)))

        # FV line (horizontal at current FV)
        fv_val = fv_weighted[tenor_sel]
        fig.add_trace(go.Scatter(x=[df_hist["dates"].iloc[0], df_hist["dates"].iloc[-1]],
                                 y=[fv_val, fv_val],
                                 mode="lines", name="FV",
                                 line=dict(color="orange", width=2)))

        # Top 5 paths by user weight
        sorted_paths = sorted(state["weights"].items(), key=lambda x: -x[1])[:5]
        colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]
        for idx, (pname, _) in enumerate(sorted_paths):
            if pname in all_fvs:
                pv = all_fvs[pname][tenor_sel]
                fig.add_trace(go.Scatter(
                    x=[df_hist["dates"].iloc[0], df_hist["dates"].iloc[-1]],
                    y=[pv, pv],
                    mode="lines", name=pname,
                    line=dict(color=colors[idx % len(colors)], dash="dot", width=1)))

        fig.update_layout(
            title=f"OIS {tenor_sel} — 1Y History + FV",
            template="plotly_white",
            height=400,
            margin=dict(l=50, r=20, t=40, b=30),
            legend=dict(font=dict(size=10)),
            yaxis_title="Rate (TNA %)",
        )
        fig.show()

# ── Wire up observers ─────────────────────────────────────────────────────────
btn_compute = widgets.Button(description="Compute", button_style="primary",
                              layout=widgets.Layout(width="120px"))
btn_compute.on_click(compute_and_display)
w_rv_select.observe(compute_and_display, names="value")
w_chart_tenor.observe(compute_and_display, names="value")
w_hedge_type.observe(compute_and_display, names="value")
w_hedge_leg1.observe(compute_and_display, names="value")
w_hedge_leg2.observe(compute_and_display, names="value")
w_hedge_dir1.observe(compute_and_display, names="value")
w_hedge_dir2.observe(compute_and_display, names="value")

# ── Layout ────────────────────────────────────────────────────────────────────
inputs_box = widgets.VBox([
    widgets.HBox([w_valdate, w_tibo] + [w_ois[t] for t in TENOR_LABELS]),
    widgets.HBox([w_notional, btn_compute]),
])

hedge_box = widgets.VBox([
    widgets.HTML("<h4>Hedge Tool</h4>"),
    widgets.HBox([w_hedge_type, w_hedge_leg1, w_hedge_leg2, w_hedge_leg3]),
    widgets.HBox([w_hedge_dir1, w_hedge_dir2, w_hedge_anchor, w_hedge_anchor_n, w_hedge_target]),
    out_hedge,
])

tab = widgets.Tab()
tab.children = [
    widgets.VBox([out_fv]),
    widgets.VBox([w_rv_select, out_spread, out_fly]),
    widgets.VBox([out_dv01, hedge_box]),
    out_maxent,
    widgets.VBox([w_chart_tenor, out_chart]),
]
tab.set_title(0, "FV & Deltas")
tab.set_title(1, "Spread/Fly RV")
tab.set_title(2, "DV01 & Hedge")
tab.set_title(3, "MaxEnt")
tab.set_title(4, "Chart")

display(widgets.HTML("<h2>PEN OIS Fair Value Model</h2>"))
display(inputs_box)
display(widgets.HTML("<h4>Rate Paths (bp changes per meeting)</h4>"))
with path_out:
    render_path_editor()
display(path_out)
display(tab)

# Auto-compute on load
compute_and_display()
print("Model ready.")
